# Projet 3 - Momentum et rotation mensuelle

A la fin de chaque mois, le portefeuille investit a poids egaux dans les trois titres ayant le meilleur rendement sur les 126 seances precedentes.

In [ ]:
from pathlib import Path
import sys

racine = Path.cwd().resolve()
while not (racine / "src" / "momentum_strategy.py").is_file() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine / "src"))

from momentum_strategy import backtest_momentum
from metrics import metriques

TICKERS = ["AI.PA", "MC.PA", "OR.PA"]
PERIODE = 126
NOMBRE_TITRES = 3
COUT = 0.001
DEBUT = "2018-01-01"

resultat = backtest_momentum(
    TICKERS,
    periode=PERIODE,
    nombre_titres=NOMBRE_TITRES,
    cout=COUT,
    debut=DEBUT,
)

In [ ]:
import pandas as pd

tableau = pd.DataFrame(
    {
        "Momentum": metriques(resultat["strategie"]),
        "Panier equilibre": metriques(resultat["rendement"]),
    }
)
display(tableau)

chemin_csv = racine / "backtest" / "projet3_momentum.csv"
chemin_csv.parent.mkdir(parents=True, exist_ok=True)
resultat.to_csv(chemin_csv)
print(f"Resultats enregistres dans {chemin_csv}")

In [ ]:
import matplotlib.pyplot as plt

courbe_momentum = (1 + resultat["strategie"]).cumprod() * 100
benchmark = (1 + resultat["rendement"]).cumprod() * 100
ax = benchmark.plot(figsize=(14, 7), color="black", linewidth=2, label="Panier equilibre")
courbe_momentum.plot(ax=ax, linewidth=2, label="Momentum mensuel")
ax.set_title("Projet 3 - Momentum sur 126 seances")
ax.set_ylabel("Valeur (base 100)")
ax.grid(True)
ax.legend()
plt.tight_layout()

display(resultat.attrs["poids"].tail(126))

Le momentum est calcule avec pct_change(126). La fin de chaque mois selectionne les trois meilleurs titres avec nlargest(3), les poids egaux sont ffill jusqu'au mois suivant, puis chaque poids est decale d'une seance pour eviter l'utilisation anticipee des donnees. Les couts sont appliques en fonction du turnover entre les poids.